# 01 · ANOVA de una vía (R)

**Semana 1 — Fundamentos y un solo factor.**

## Objetivos
- Ajustar un ANOVA de una vía para un experimento de un solo factor.
- Verificar los supuestos mediante el análisis de residuales.
- Comparar tratamientos con **Tukey HSD** y, frente a un control, con **Dunnett**.

## Paquetes
`car` (supuestos), `multcomp` y `emmeans` (comparaciones múltiples).

> Teoría: [`../teoria/03-anova-una-via.md`](../teoria/03-anova-una-via.md) y [`../teoria/04-comparaciones-multiples.md`](../teoria/04-comparaciones-multiples.md).

In [ ]:
suppressMessages({
  library(car)
  library(multcomp)
  library(emmeans)
})
set.seed(1)  # reproducibilidad

## 1. Los datos

Resistencia de una fibra sintética según el **% de algodón** (5 niveles, 5 réplicas; Montgomery, ej. 3.1).

In [ ]:
df <- read.csv('../datos/resistencia-algodon.csv')
df$algodon_pct <- factor(df$algodon_pct)
aggregate(resistencia ~ algodon_pct, data = df,
          FUN = function(x) c(media = mean(x), sd = sd(x)))

In [ ]:
boxplot(resistencia ~ algodon_pct, data = df,
        xlab = '% algodón', ylab = 'Resistencia',
        main = 'Resistencia por % de algodón', col = 'lightsteelblue')

## 2. ANOVA de una vía

Modelo $y_{ij} = \mu + \tau_i + \varepsilon_{ij}$, con $H_0: \tau_i = 0\ \forall i$.

In [ ]:
modelo <- aov(resistencia ~ algodon_pct, data = df)
summary(modelo)
cat('R2 =', summary(lm(modelo))$r.squared, '\n')

Valor-p $\approx 9\times10^{-6}$: **se rechaza $H_0$** ($R^2\approx0.75$).

## 3. Verificación de supuestos (residuales)

In [ ]:
op <- par(mfrow = c(1, 2))
qqnorm(residuals(modelo)); qqline(residuals(modelo))
plot(fitted(modelo), residuals(modelo),
     xlab = 'Ajustados', ylab = 'Residuales',
     main = 'Residuales vs. ajustados'); abline(h = 0, lty = 2)
par(op)

In [ ]:
shapiro.test(residuals(modelo))                   # normalidad
leveneTest(resistencia ~ algodon_pct, data = df)  # homocedasticidad (car)

Shapiro-Wilk ($p\approx0.18$) y Levene ($p\approx0.86$) no rechazan: supuestos OK.

## 4. Comparaciones múltiples: Tukey HSD

In [ ]:
TukeyHSD(modelo, conf.level = 0.95)
plot(TukeyHSD(modelo))

El nivel **30 %** da la mayor resistencia y supera a 15, 20 y 35 %.

## 5. Comparaciones contra un control: Dunnett

Tiempos de coagulación según **dieta** (A = control). Usamos `multcomp::glht`.

In [ ]:
coag <- read.csv('../datos/tiempo-coagulacion.csv')
coag$dieta <- relevel(factor(coag$dieta), ref = 'A')
mod_c <- aov(tiempo ~ dieta, data = coag)
dunnett <- glht(mod_c, linfct = mcp(dieta = 'Dunnett'))
summary(dunnett)
confint(dunnett)

Frente al control A, **B y C** difieren significativamente; **D** no.

> Alternativa con `emmeans`:
> ```r
> emm <- emmeans(mod_c, ~ dieta)
> contrast(emm, method = 'trt.vs.ctrl', ref = 'A')
> ```

## 6. Conclusiones

- El % de algodón **afecta** la resistencia (ANOVA, $p\approx 9\times10^{-6}$).
- Supuestos verificados.
- **Tukey:** 30 % es el mejor nivel.
- **Dunnett:** B y C superan al control A; D no.

> Resultados equivalentes a la versión en Python ([`01-anova-una-via_py.ipynb`](01-anova-una-via_py.ipynb)).

---

## Ejemplo 2 – Diferente dominio: rendimiento de cultivo por fertilizante

Rendimiento (kg/ha) bajo **cuatro fertilizantes** (A–D), 5 réplicas cada uno. Mismo flujo ANOVA aplicado a un contexto agrícola.

In [ ]:
fert <- data.frame(
  fertilizante = factor(rep(c('A','B','C','D'), each = 5)),
  rendimiento  = c(18, 20, 19, 22, 21,
                   25, 27, 26, 28, 24,
                   22, 23, 21, 24, 23,
                   16, 17, 15, 18, 16)
)
boxplot(rendimiento ~ fertilizante, data = fert,
        xlab = 'Fertilizante', ylab = 'Rendimiento (kg/ha)',
        main = 'Rendimiento por fertilizante', col = 'palegreen3')
aggregate(rendimiento ~ fertilizante, data = fert,
          FUN = function(x) c(media = mean(x), sd = round(sd(x), 2)))

In [ ]:
mod_f <- aov(rendimiento ~ fertilizante, data = fert)
summary(mod_f)
cat('R2 =', round(summary(lm(mod_f))$r.squared, 4), '\n')

In [ ]:
op <- par(mfrow = c(1, 2))
qqnorm(residuals(mod_f)); qqline(residuals(mod_f))
plot(fitted(mod_f), residuals(mod_f),
     xlab = 'Ajustados', ylab = 'Residuales',
     main = 'Residuales vs. ajustados'); abline(h = 0, lty = 2)
par(op)
shapiro.test(residuals(mod_f))
leveneTest(rendimiento ~ fertilizante, data = fert)

Supuestos verificados ($p$-Shapiro $>0.05$, $p$-Levene $>0.05$). Procedemos con Tukey HSD.

In [ ]:
TukeyHSD(mod_f)
plot(TukeyHSD(mod_f))

**B** supera significativamente a todos los demás fertilizantes; **D** da el menor rendimiento y difiere de A, B y C.

---

## Ejemplo 3 – Supuestos incumplidos: defectos por operario

Número de defectos diarios producidos por **cinco operarios** (5 días cada uno). Los datos de conteo son típicamente asimétricos y heterocedásticos: un caso donde los supuestos del ANOVA fallan.

In [ ]:
defectos <- data.frame(
  operario   = factor(rep(paste0('Op', 1:5), each = 5)),
  n_defectos = c( 1,  2,  1,  2,  1,
                  4,  8,  5, 12,  6,
                 20, 35, 25, 45, 30,
                  2,  1,  3,  2,  1,
                  8, 15, 10, 20, 12)
)
boxplot(n_defectos ~ operario, data = defectos,
        xlab = 'Operario', ylab = 'Defectos diarios',
        main = 'Defectos diarios por operario', col = 'lightsalmon')
aggregate(n_defectos ~ operario, data = defectos,
          FUN = function(x) c(media = round(mean(x), 1), sd = round(sd(x), 1)))

In [ ]:
mod_d <- aov(n_defectos ~ operario, data = defectos)
summary(mod_d)
op <- par(mfrow = c(1, 2))
qqnorm(residuals(mod_d)); qqline(residuals(mod_d))
plot(fitted(mod_d), residuals(mod_d),
     xlab = 'Ajustados', ylab = 'Residuales',
     main = 'Residuales vs. ajustados'); abline(h = 0, lty = 2)
par(op)
shapiro.test(residuals(mod_d))
leveneTest(n_defectos ~ operario, data = defectos)

Levene ($p < 0.05$): **varianzas heterogéneas**; Shapiro-Wilk rechaza normalidad. El ANOVA clásico **no es apropiado** para estos datos. Alternativas:

- **Transformación** $\sqrt{y}$ o $\log(y+1)$ para estabilizar la varianza.
- **Kruskal-Wallis**: prueba no paramétrica equivalente (ver `02-no-parametricas-transformaciones`).

In [ ]:
kruskal.test(n_defectos ~ operario, data = defectos)

Kruskal-Wallis confirma diferencias significativas entre operarios sin asumir normalidad ni homocedasticidad.

---

## Ejemplo 4 – Dataset real: `PlantGrowth`

Dataset `PlantGrowth` (incluido en R base): peso seco de plantas bajo un **control** y dos **tratamientos** (trt1, trt2), 10 réplicas por grupo (Dobson, 1983). Ilustra que un ANOVA global significativo no implica que *todas* las comparaciones por pares sean significativas.

In [ ]:
data(PlantGrowth)
boxplot(weight ~ group, data = PlantGrowth,
        xlab = 'Grupo', ylab = 'Peso seco (g)',
        main = 'PlantGrowth: peso por tratamiento', col = 'lightgreen')
aggregate(weight ~ group, data = PlantGrowth,
          FUN = function(x) c(media = round(mean(x), 3), sd = round(sd(x), 3)))

In [ ]:
mod_pg <- aov(weight ~ group, data = PlantGrowth)
summary(mod_pg)
cat('R2 =', round(summary(lm(mod_pg))$r.squared, 4), '\n')

In [ ]:
op <- par(mfrow = c(1, 2))
qqnorm(residuals(mod_pg)); qqline(residuals(mod_pg))
plot(fitted(mod_pg), residuals(mod_pg),
     xlab = 'Ajustados', ylab = 'Residuales',
     main = 'Residuales vs. ajustados'); abline(h = 0, lty = 2)
par(op)
shapiro.test(residuals(mod_pg))
leveneTest(weight ~ group, data = PlantGrowth)

In [ ]:
TukeyHSD(mod_pg)
plot(TukeyHSD(mod_pg))

Solo **trt1 vs. trt2** resulta significativo ($p\approx0.01$); trt2–ctrl roza la significancia ($p\approx0.09$). Lección: el $p$-valor global del ANOVA no garantiza que todas las comparaciones por pares lo sean.